In [1]:
import os
import warnings
import logging

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.validation import check_is_fitted
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/tcn-script/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
import importlib

from scipy.stats import randint, uniform, loguniform

tf.keras.backend.clear_session()

E0000 00:00:1775867304.051371      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775867304.147615      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775867305.016766      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775867305.016809      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775867305.016812      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775867305.016815      23 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_run_name = 'tcn_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
pipe_name = 'temporal_extractor'

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
random_search = True

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=0.2,
    test_pct=0.2
)

Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
importlib.reload(utils)

raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns
)

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), make_column_selector(pattern='acc|rot|thm|tof')),
], remainder='drop', verbose_feature_names_out=False)
preprocessor.set_output(transform='pandas')

sequence_builder = utils.SequencePadder(maxlen=80, padding_value=-999.0)

tcn_clf = utils.KerasTCNClassifier()

classifier = utils.ManyToOneWrapperRNN(estimator=tcn_clf, target='gesture_action')

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ('preprocessor', preprocessor),
    ('sequence_builder', sequence_builder),
    ('classifier', classifier),
])

cv = GroupKFold(n_splits=n_splits)

In [6]:
if random_search:
    param_grid = {
        # RawSequenceExtractor
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [tof_columns],
        f'{pipe_name}__acc_mode':      ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode':      ['centered'],
        f'{pipe_name}__tof_mode':      ['baseline'],

        # SequencePadder
        'sequence_builder__maxlen':         [10, 20, 35, 50, 60, 80, 120],
        'sequence_builder__padding_value':  [-999.0],

        # ============ TCN ============
        'classifier__estimator__nb_filters': [64, 128],  # Uniform integer
        'classifier__estimator__kernel_size': [2, 5, 9],  # Uniform integer 2-8
        'classifier__estimator__nb_stacks': [1, 3, 5],  # Uniform integer 1-4
        'classifier__estimator__dilations': [  # Can't use distribution, use choices
            (1, 2, 4, 8),
            (1, 2, 4, 8, 16),
            (1, 2, 4, 8, 16, 32),
            (1, 2, 4, 8, 16, 32, 64),
        ],
        'classifier__estimator__use_skip_connections': [True],
        'classifier__estimator__dense_units': [  # Discrete choices
                      (32,), (64,), 
            (16, 8), (32, 16), (64, 32)
        ],
        'classifier__estimator__dropout': uniform(0.1, 0.4),  # Uniform 0.1-0.5
        'classifier__estimator__learning_rate': loguniform(1e-4, 1e-2),  # Log uniform
        'classifier__estimator__batch_size': randint(8, 65),  # Uniform 8-64
        'classifier__estimator__patience': randint(10, 21),  # Uniform 10-20
    }
else:
    param_grid = {
        # RawSequenceExtractor
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [tof_columns],
        f'{pipe_name}__acc_mode':      ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode':      ['centered'],
        f'{pipe_name}__tof_mode':      ['baseline'],

        # SequencePadder
        'sequence_builder__maxlen':         [80],
        'sequence_builder__padding_value':  [-999.0],

        # TCN hyperparameters
        'classifier__estimator__nb_filters':          [32],
        'classifier__estimator__kernel_size':         [3],
        'classifier__estimator__nb_stacks':           [1],
        'classifier__estimator__dilations':           [(1, 2, 4, 8)],
        'classifier__estimator__use_skip_connections':[True],
        'classifier__estimator__dense_units':         [(32,)],
        'classifier__estimator__dropout':             [0.2],
        'classifier__estimator__learning_rate':       [1e-3],
        'classifier__estimator__batch_size':          [16],
        'classifier__estimator__epochs':              [100],
        'classifier__estimator__patience':            [10],
        'classifier__estimator__class_weight_mode':   ['balanced'],
    }

In [7]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=5,
                cv=cv,
                random_state=42,
                n_jobs=1,
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=1,
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']

        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)

    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

Fitting 3 folds for each of 5 candidates, totalling 15 fits


I0000 00:00:1775867369.952985      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775867369.959053      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1775867401.715165      70 service.cc:152] XLA service 0x7ec5d0002e80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775867401.715204      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775867401.715208      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775867407.758217      70 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775867439.577576      70 device_compiler.h:188] Compiled clust

In [8]:
master_cv_results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier__estimator__batch_size,param_classifier__estimator__dense_units,param_classifier__estimator__dilations,param_classifier__estimator__dropout,param_classifier__estimator__kernel_size,param_classifier__estimator__learning_rate,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score,orientation_data,model_target
0,116.871526,4.662884,9.577919,0.159100,46,"(32, 16)","(1, 2, 4, 8)",0.173374,2,0.001562,...,0.272370,0.099482,2,0.277228,0.17,0.237113,0.228114,0.044236,Lie on Back,gesture_action
1,124.524217,4.500855,10.725438,0.158627,18,"(64, 32)","(1, 2, 4, 8, 16, 32, 64)",0.157147,9,0.000110,...,0.310374,0.051120,1,0.693069,0.36,0.752577,0.601882,0.172753,Lie on Back,gesture_action
2,213.401611,18.992672,18.222665,0.149373,28,"(32,)","(1, 2, 4, 8, 16, 32, 64)",0.344661,2,0.000731,...,0.254960,0.023350,3,0.326733,0.27,0.412371,0.336368,0.058521,Lie on Back,gesture_action
3,206.549483,3.872787,16.727286,0.337871,49,"(32, 16)","(1, 2, 4, 8, 16, 32, 64)",0.246545,5,0.000152,...,0.200397,0.044099,5,0.405941,0.33,0.412371,0.382771,0.037407,Lie on Back,gesture_action
4,31.055476,1.080615,3.953797,0.073198,64,"(16, 8)","(1, 2, 4, 8)",0.343018,2,0.000796,...,0.232165,0.080901,4,0.346535,0.36,0.319588,0.342041,0.016802,Lie on Back,gesture_action
